#   작업
1. 장소 검색한다.
 • 내 위치를 기반으로 주변 장소를 검색한다. <br>
 • 장소명 또는 카테고리 (식당, 카페, 관광지, 숙소, ...)로 검색한다.<br>
 • 주소로 검색한다.<br>
 • 검색된 장소의 목록에는 장소명, 설명, 평점, 리뷰수를 포함한다.<br>
 • 장소를 선택하면, 그 장소에 달린 리뷰 목록을 출력한다.<br>
 • 장소에 북마크를 하여 내 장소로 관리한다.<br>
 • 장소에 리뷰와 평점을 작성한다
<br>
<br>
근데 저거 데이터 만들기 다 귀찮은데 
## 자동화
1. 네이버 지도 웹 크롤링 <br>
2. 동네 선정 해서 카테고리 (ex.식당, 편의점, 카페, 디저트,숙소) 주소 가져오자.<br>
3. kakao api 사용해서 좌표 쫙 찍어오자 <br>

    place_id INT AUTO_INCREMENT PRIMARY KEY,
    x FLOAT,
    y FLOAT,
    name VARCHAR(20),
    description VARCHAR(200),
    category VARCHAR(100),
    address VARCHAR(100)

-이중에 x,y 에는 좌표 넣고 , name 에는 가계 이름 넣고, discription에는 뭐넣지.., 카테고리는 카테고리 넣고 address 에는 주소 넣게 코드를 짜보자

In [1]:
! pip install pymysql
! pip install cryptography

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: C:\Program Files\새 폴더\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: C:\Program Files\새 폴더\python.exe -m pip install --upgrade pip


In [ ]:
import time
import json
import requests
import pymysql
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from urllib.parse import quote

In [3]:
# ============================
# 1) kakao_config.json에서 API 키 읽어오기
# ============================
def load_kakao_key(config_path="kakao_config.json"):
    """
    kakao_config.json 파일에서 Kakao REST API 키를 읽어서 반환합니다.
    """
    try:
        with open(config_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            key = data.get("KAKAO_REST_API_KEY")
            if not key:
                raise KeyError(f"'KAKAO_REST_API_KEY' 키가 {config_path}에 없습니다.")
            return key
    except FileNotFoundError:
        print(f"[ERROR] {config_path} 파일을 찾을 수 없습니다.")
        return None
    except json.JSONDecodeError:
        print(f"[ERROR] {config_path} 파일이 올바른 JSON 형식이 아닙니다.")
        return None


KAKAO_REST_API_KEY = load_kakao_key()
if KAKAO_REST_API_KEY is None:
    raise SystemExit("API 키를 불러오지 못해 프로그램을 종료합니다.")

In [4]:
# ============================
# 2) db_config.json에서 MySQL 연결 정보 읽어오기
# ============================
def load_db_config(config_path="db_config.json"):
    with open(config_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        return {
            "host": data["host"],
            "user": data["user"],
            "password": data["password"],
            "db": data["db"],
            "charset": data.get("charset", "utf8mb4"),
            "cursorclass": pymysql.cursors.DictCursor
        }


DB_CONFIG = load_db_config()
if not DB_CONFIG:
    raise SystemExit("db_config.json에서 DB 설정을 불러올 수 없습니다.")

In [ ]:
driver_path = 

In [13]:
# # ============================
# # 3) Selenium Chrome WebDriver 설정
# # ============================
# chrome_options = Options()
# chrome_options.add_argument("--headless")
# chrome_options.add_argument("--no-sandbox")
# chrome_options.add_argument("--disable-gpu")

# CHROME_DRIVER_PATH = r"C:\Users\charl\OneDrive\Desktop\기타\coding\프로그램\chromedriver-win64\chromedriver.exe"

# # Service 객체 생성
# service = Service(CHROME_DRIVER_PATH)

# # driver 초기화 시 executable_path 대신 service=Service(...) 사용
# driver = webdriver.Chrome(service=service, options=chrome_options)
# driver.implicitly_wait(5)


chrome_options = Options()
# # GUI 창을 보려면 아래 두 줄을 주석 처리하세요
# chrome_options.add_argument("--headless")
# chrome_options.add_argument("--disable-gpu")

# Windows 환경에서 창 최소화로 실행하고 싶다면 이렇게 추가
# chrome_options.add_argument("--start-minimized")
# chrome_options.add_argument("--window-size=1920,1080")

service = Service(ChromeDriverManager().install(), port=0)
driver = webdriver.Chrome(service=service, options=chrome_options)
driver.implicitly_wait(5)

In [14]:
# ============================
# 4) 동네 및 카테고리 설정
# ============================
NEIGHBORHOOD = "서울 마포구"
CATEGORIES = [
    ("음식점", "음식점"),
    ("카페", "카페"),
    ("숙박", "숙박"),
    ("주유정보", "주유소")  # “주유정보(주유소 전체)”라고 카카오맵에서 표시됨
]

In [16]:
# ============================
# 5) Kakao API로 주소 → 위·경도 변환 함수
# ============================


def get_coordinates_from_kakao(address: str):
    """
    Kakao Local API(Address Search)를 호출해서
    주어진 주소의 x(경도), y(위도)를 반환합니다.
    """
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": KAKAO_REST_API_KEY}
    params = {"query": address}
    resp = requests.get(url, headers=headers, params=params)
    if resp.status_code != 200:
        print(
            f"[Kakao API ERROR] status_code={resp.status_code}, address={address}")
        return None, None

    data = resp.json()
    documents = data.get("documents")
    if not documents:
        return None, None

    x = float(documents[0]["x"])
    y = float(documents[0]["y"])
    return x, y

In [17]:
# ============================
# 6) DB INSERT 함수
# ============================

def insert_place_into_db(cursor, x, y, name, description, category, address):
    sql = """
    INSERT INTO place (x, y, name, description, category, address)
    VALUES (%s, %s, %s, %s, %s, %s);
    """
    cursor.execute(sql, (x, y, name, description, category, address))

In [18]:
# def crawl_naver_map_places(neighborhood: str, category: str):
#     """
#     1) https://map.naver.com/v5/search/{검색어} 로 접속
#     2) iframe 내부에 있는 검색 결과에서 name, address, category, description 추출
#     3) 반환값: [(name, address, description, category), ...]
#     """
#     query = f"{neighborhood} {category}"
#     url = f"https://map.naver.com/v5/search/{query}"
#     driver.get(url)
#     time.sleep(2)

#     # 검색 결과가 들어 있는 iframe으로 전환
#     try:
#         driver.switch_to.frame("searchIframe")
#     except:
#         all_iframes = driver.find_elements(By.TAG_NAME, "iframe")
#         switched = False
#         for iframe in all_iframes:
#             try:
#                 driver.switch_to.frame(iframe)
#                 if driver.find_elements(By.CSS_SELECTOR, ".place_item"):
#                     switched = True
#                     break
#             except:
#                 driver.switch_to.default_content()
#         if not switched:
#             print(f"[ERROR] '{query}' 검색 결과 iframe 전환 실패.")
#             return []

#     time.sleep(1)

#     places = []
#     # (DOM 구조가 바뀔 수 있으므로, 실제 개발자 도구로 확인 후 selector를 수정하세요)
#     item_list = driver.find_elements(By.CSS_SELECTOR, "div.place_item")
#     if not item_list:
#         item_list = driver.find_elements(By.CSS_SELECTOR, "li._3XS2T")

#     for item in item_list:
#         try:
#             name_elem = item.find_element(By.CSS_SELECTOR, "a.place_bluelink")
#             name = name_elem.text.strip()

#             try:
#                 cat_elem = item.find_element(By.CSS_SELECTOR, "span._3ocDE")
#                 category_text = cat_elem.text.strip()
#             except:
#                 category_text = category

#             try:
#                 addr_elem = item.find_element(By.CSS_SELECTOR, "span._2yqUQ")
#                 address = addr_elem.text.strip()
#             except:
#                 address = ""

#             try:
#                 desc_elem = item.find_element(By.CSS_SELECTOR, "li._2j_R6")
#                 description = desc_elem.text.strip()
#             except:
#                 description = ""

#             if name and address:
#                 places.append((name, address, description, category_text))
#         except:
#             continue

#     driver.switch_to.default_content()
#     return places

In [19]:
# ============================
# 7) Kakao Map 크롤링 함수 
# ============================

def crawl_kakao_map_places(neighborhood: str, category: str, limit: int = 10):
    """
    1) https://map.kakao.com/?q={검색어} 로 접속
    2) 검색 결과 리스트(item_list)에서 name, address, category_text, description 추출
    3) 최대 'limit'개 만큼만 반환
    """
    query = f"{neighborhood} {category}"
    # URL 인코딩
    url = f"https://map.kakao.com/?q={quote(query)}"
    driver.get(url)
    time.sleep(2)  # 최소한 페이지가 로드될 시간 확보

    # → Kakao Map은 iframe을 사용하지 않으므로 switch_to.frame은 필요 없습니다.
    #    단, 콘텐츠가 Ajax로 로드되므로 약간의 sleep 또는 ExplicitWait이 필요합니다.
    time.sleep(1)

    places = []
    # (아래 CSS Selector는 예시입니다. 반드시 실제 개발자 도구에서 확인 후 수정하세요!)
    # Kakao Map 검색 결과 목록은 ul.tag_placelist > li.PlaceItem 형태로 되어 있음
    item_list = driver.find_elements(
        By.CSS_SELECTOR, "ul.placelist li.PlaceItem")

    # 만약 위 Selector로 나오지 않는다면, “li.PlaceItem” 혹은 “.PlaceItem” 등으로 수정하세요.
    if not item_list:
        item_list = driver.find_elements(By.CSS_SELECTOR, "li.PlaceItem")

    count = 0
    for item in item_list:
        if count >= limit:
            break

        try:
            # (1) 장소명
            #   - 일반적으로 <a class="link_name">장소명</a> 형태
            name_elem = item.find_element(By.CSS_SELECTOR, "a.link_name")
            name = name_elem.text.strip()
        except:
            continue

        # (2) 카테고리 텍스트
        try:
            # <span class="category">음식점, 한식</span> 형태일 수 있음
            cat_elem = item.find_element(By.CSS_SELECTOR, "span.category")
            category_text = cat_elem.text.strip()
        except:
            # CSS selector가 바뀌었거나, 카테고리 정보가 없으면
            category_text = category

        # (3) 주소
        try:
            # <p class="addr">서울 마포구 ...</p> 형태
            addr_elem = item.find_element(By.CSS_SELECTOR, "p.addr")
            address = addr_elem.text.strip()
        except:
            address = ""

        # (4) 간단 설명(description) — Kakao Map 검색 결과 목록에는
        #     추가적인 설명이 항상 붙어있는 건 아니므로,
        #     “” 또는 점포 설명이 따로 있으면 가져오도록 시도
        try:
            # 예: <p class="desc">브런치, 디저트, 커피</p> 같은 형식
            desc_elem = item.find_element(By.CSS_SELECTOR, "p.desc")
            description = desc_elem.text.strip()
        except:
            description = ""

        # (5) 위/경도(x, y) – Kakao API로 변환
        if address:
            x, y = get_coordinates_from_kakao(address)
        else:
            x, y = None, None

        # (6) 유효값 검사: name과 address, 위/경도 둘 다 있으면 추가
        if name and address and x is not None and y is not None:
            places.append((name, address, description, category_text, x, y))
            count += 1

    return places

In [ ]:
# ============================
# 8) 메인 로직: 크롤링 → 좌표 변환 → DB INSERT
# ============================

def main():
    conn = pymysql.connect(**DB_CONFIG)
    try:
        with conn.cursor() as cursor:
            for cat_keyword, cat_label in CATEGORIES:
                print(f"\n=== '{NEIGHBORHOOD} {cat_keyword}' 크롤링 시작 ===")
                # limit=10 → 각 카테고리별로 최대 10개 장소 수집
                place_list = crawl_kakao_map_places(
                    NEIGHBORHOOD, cat_keyword, limit=10)
                print(f"총 {len(place_list)}개 장소를 찾았습니다.")

                # DB에 삽입
                for idx, (name, address, description, category_text, x, y) in enumerate(place_list, start=1):
                    insert_place_into_db(
                        cursor, x, y, name, description, category_text, address)
                    print(
                        f"  [{idx}/{len(place_list)}] INSERT: {name} ({address}) → (x={x}, y={y})")

                conn.commit()
                print(f"=== '{cat_keyword}' 카테고리 INSERT 커밋 완료 ===")

        print("모든 작업이 완료되었습니다.")
    except Exception as e:
        print("예외 발생:", e)
        conn.rollback()
    # finally:
    #     conn.close()
    #     driver.quit()


if __name__ == "__main__":
    main()


=== '서울 마포구 음식점' 크롤링 시작 ===
총 0개 장소를 찾았습니다.
=== '음식점' 카테고리 INSERT 커밋 완료 ===

=== '서울 마포구 카페' 크롤링 시작 ===
총 0개 장소를 찾았습니다.
=== '카페' 카테고리 INSERT 커밋 완료 ===

=== '서울 마포구 숙박' 크롤링 시작 ===
총 0개 장소를 찾았습니다.
=== '숙박' 카테고리 INSERT 커밋 완료 ===

=== '서울 마포구 주유정보' 크롤링 시작 ===
총 0개 장소를 찾았습니다.
=== '주유정보' 카테고리 INSERT 커밋 완료 ===
모든 작업이 완료되었습니다.


In [22]:
import time
import json
import requests
import pymysql
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from urllib.parse import quote

# ============================
# 1) kakao_config.json에서 API 키 읽어오기
# ============================


def load_kakao_key(config_path="kakao_config.json"):
    with open(config_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        return data.get("KAKAO_REST_API_KEY")


KAKAO_REST_API_KEY = load_kakao_key()
if not KAKAO_REST_API_KEY:
    raise SystemExit("kakao_config.json에서 Kakao API 키를 불러올 수 없습니다.")

# ============================
# 2) db_config.json에서 MySQL 연결 정보 읽어오기
# ============================


def load_db_config(config_path="db_config.json"):
    with open(config_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        return {
            "host": data["host"],
            "user": data["user"],
            "password": data["password"],
            "db": data["db"],
            "port": data.get("port", 3306),
            "charset": data.get("charset", "utf8mb4"),
            "cursorclass": pymysql.cursors.DictCursor
        }


DB_CONFIG = load_db_config()
if not DB_CONFIG:
    raise SystemExit("db_config.json에서 DB 설정을 불러올 수 없습니다.")

# ============================
# 3) Selenium Chrome WebDriver 설정
# ============================
chrome_options = Options()
# GUI 창을 보고 싶다면 아래 두 줄을 주석 처리하세요.
# chrome_options.add_argument("--headless")
# chrome_options.add_argument("--disable-gpu")

# webdriver-manager를 사용해 자동으로 현재 Chrome 버전에 맞는 드라이버를 설치
service = Service(ChromeDriverManager().install(), port=0)
driver = webdriver.Chrome(service=service, options=chrome_options)
driver.implicitly_wait(5)  # 내부적으로 최대 5초 대기

# ============================
# 4) Kakao API: 주소 → (경도 x, 위도 y) 변환 함수
# ============================


def get_coordinates_from_kakao(address: str):
    """
    Kakao Local API(Address Search)를 호출해 주소를 x, y로 변환
    """
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": KAKAO_REST_API_KEY}
    params = {"query": address}
    resp = requests.get(url, headers=headers, params=params)
    if resp.status_code != 200:
        print(
            f"[Kakao API ERROR] status_code={resp.status_code}, address={address}")
        return None, None

    data = resp.json()
    documents = data.get("documents")
    if not documents:
        return None, None

    x = float(documents[0]["x"])
    y = float(documents[0]["y"])
    return x, y

# ============================
# 5) DB INSERT 함수
# ============================


def insert_place_into_db(cursor, x, y, name, description, category, address):
    sql = """
    INSERT INTO place (x, y, name, description, category, address)
    VALUES (%s, %s, %s, %s, %s, %s);
    """
    cursor.execute(sql, (x, y, name, description, category, address))

# ============================
# 6) Kakao Map에서 “서울 마포구 서교동” 선택 & 카테고리별 크롤링 함수
# ============================


def crawl_kakao_map_places(category_name: str, limit: int = 10):
    """
    사전에 driver.get("https://map.kakao.com/")가 되어 있다고 가정.
    1) 팝업으로 떠 있는 “로컬 선택”에서
       - “서울특별시” → “마포구” → “서교동” 순서대로 클릭해서 동(서교동)을 선택
    2) 왼쪽 상단의 카테고리 버튼 중, category_name(예: “음식점”, “카페”, “숙박”, “주유소”) 클릭
    3) 검색 결과 리스트에서 최대 limit개(place_item)만큼
       (name, address, description, category, x, y)를 추출 후 반환
    """
    # 1) 이미 base 페이지를 띄운 상태라면 아래 코드를 통해 동 선택을 진행
    #    (처음 한 번만 실행하면, 이후에는 브라우저를 재시작하지 않는 한 선택 상태가 유지될 수 있음)
    #   — 만약 매번 새롭게 드라이버를 킨 상태라면, 이 함수 호출 전에
    #     driver.get("https://map.kakao.com/")를 해 주세요.

    # ▶ 1-1) “서울특별시” 클릭
    province_btn = driver.find_element(
        By.CSS_SELECTOR, "a#localInfo\\.map\\.province.selectBox")
    province_btn.click()
    time.sleep(0.5)

    #     팝업 내부의 <div class="province"> 에서 “서울특별시” 텍스트를 가진 <a> 태그 클릭
    driver.find_element(
        By.XPATH,
        "//div[@class='province']//li/a[text()='서울특별시']"
    ).click()
    time.sleep(0.5)

    # ▶ 1-2) “마포구” 클릭
    county_btn = driver.find_element(
        By.CSS_SELECTOR, "a#localInfo\\.map\\.county.selectBox")
    county_btn.click()
    time.sleep(0.5)

    driver.find_element(
        By.XPATH,
        "//div[@class='county']//li/a[text()='마포구']"
    ).click()
    time.sleep(0.5)

    # ▶ 1-3) “서교동” 클릭
    town_btn = driver.find_element(
        By.CSS_SELECTOR, "a#localInfo\\.map\\.town.selectBox.ACTIVE, a#localInfo\\.map\\.town.selectBox")
    # — 클래스명이 “selectBox ACTIVE” 로 설정되는 경우도 있고, 처음엔 “selectBox”만 있을 수 있으므로 두 가지를 모두 잡을 수 있게 콤마(,)로 처리
    town_btn.click()
    time.sleep(0.5)

    driver.find_element(
        By.XPATH,
        "//div[@class='town']//li/a[text()='서교동']"
    ).click()
    time.sleep(1)  # “서교동” 선택 후, 좌측 패널이 동네 정보로 갱신될 시간

    # 2) 왼쪽 “주변 탐색” 영역의 카테고리 버튼 누르기
    #    카테고리 버튼들은 title 속성에 한글 카테고리명(“음식점”, “카페”, “숙박”, “주유소” 등)이 들어 있음
    try:
        cat_btn = driver.find_element(
            By.XPATH, f"//button[@title='{category_name}']")
        cat_btn.click()
    except:
        print(f"[ERROR] '{category_name}' 버튼을 찾지 못했습니다.")
        return []
    time.sleep(2)  # 카테고리 선택 후 검색 결과가 로드될 시간

    # 3) 검색 결과 리스트(item_list)에서 최대 limit개 추출
    places = []
    count = 0

    # 3-1) 검색 결과는 ul.placelist > li.PlaceItem 형식 (버전별로 클래스/태그가 달라질 수 있으므로 확인 필수)
    item_list = driver.find_elements(
        By.CSS_SELECTOR, "ul.placelist li.PlaceItem")
    if not item_list:
        # 혹시 그냥 li.PlaceItem만 잡힌다면 fallback
        item_list = driver.find_elements(By.CSS_SELECTOR, "li.PlaceItem")

    for item in item_list:
        if count >= limit:
            break

        try:
            # — (1) 장소명: <a class="link_name"> … </a>
            name_elem = item.find_element(By.CSS_SELECTOR, "a.link_name")
            name = name_elem.text.strip()
        except:
            continue

        # — (2) 주소: <div class="addr"> <p>도로명 주소</p> <p>상세 설명</p> … </div>
        try:
            address = item.find_element(
                By.CSS_SELECTOR, "div.addr > p:nth-child(1)").text.strip()
        except:
            address = ""

        # — (3) 카테고리명: “음식점”, “카페”, “숙박”, “주유소” 등
        #       (아예 검색할 때 이미 눌러준 category_name을 그대로 사용해도 되지만,
        #        혹시 세부 카테고리가 다르게 나오는 경우를 대비해 리스트 내 텍스트를 읽어 와도 좋습니다.)
        try:
            category_text = item.find_element(
                By.CSS_SELECTOR, "span.category").text.strip()
        except:
            # 만약 Kakao Map이 상세 카테고리를 span.category로 제공하지 않으면,
            # 우리가 클릭한 category_name을 기본값으로 둠
            category_text = category_name

        # — (4) 간단 설명(description): <p class="desc"> … </p> (있으면, 없으면 빈 문자열)
        try:
            description = item.find_element(
                By.CSS_SELECTOR, "p.desc").text.strip()
        except:
            description = ""

        # — (5) 위/경도(x, y) – Kakao 주소 검색 API 사용
        if address:
            x, y = get_coordinates_from_kakao(address)
        else:
            x, y = None, None

        # — (6) name, address, x, y가 모두 존재해야 DB에 넣음
        if name and address and x is not None and y is not None:
            places.append((name, address, description, category_text, x, y))
            count += 1

    return places


# ============================
# 7) 동네/카테고리 설정
# ============================
NEIGHBORHOOD_FULL = ("서울특별시", "마포구", "서교동")
# Kakao Map “주변 탐색” 버튼의 title 속성값과 동일하게 지정
CATEGORIES = [
    "음식점",   # 음식점(전체)
    "카페",     # 카페
    "숙박",     # 숙박(전체)
    "주유소"    # 주유정보 → “주유소” 아이콘 클릭
]

# ============================
# 8) 메인 로직: 동네 선택 → 카테고리별 크롤링 → DB INSERT
# ============================


def main():
    # 0) 먼저 Kakao Map 기본 페이지로 이동
    driver.get("https://map.kakao.com/")
    time.sleep(1)

    # MySQL 연결
    conn = pymysql.connect(**DB_CONFIG)
    try:
        with conn.cursor() as cursor:
            # 1) “서울특별시-마포구-서교동” 선택은
            #    crawl_kakao_map_places() 함수 안에서 수행(매 카테고리마다 같은 동을 누르게 됨)

            for cat in CATEGORIES:
                print(
                    f"\n=== '{NEIGHBORHOOD_FULL[1]} {NEIGHBORHOOD_FULL[2]} {cat}' 크롤링 시작 ===")
                place_list = crawl_kakao_map_places(cat, limit=10)
                print(f"총 {len(place_list)}개 장소를 찾았습니다.")

                # 2) 결과를 한 건씩 DB에 INSERT
                for idx, (name, address, description, category_text, x, y) in enumerate(place_list, start=1):
                    insert_place_into_db(
                        cursor, x, y, name, description, category_text, address)
                    print(
                        f"  [{idx}/{len(place_list)}] INSERT: {name} | {address} | ({x}, {y})")

                conn.commit()
                print(f"=== '{cat}' 카테고리 INSERT 커밋 완료 ===")

        print("모든 카테고리 작업이 완료되었습니다.")
    except Exception as e:
        print("예외 발생:", e)
        conn.rollback()
    finally:
        conn.close()
        driver.quit()


if __name__ == "__main__":
    main()


=== '마포구 서교동 음식점' 크롤링 시작 ===
예외 발생: Message: element click intercepted: Element <a id="localInfo.map.province" href="#" class="selectBox">...</a> is not clickable at point (557, 28). Other element would receive the click: <div id="dimmedLayer" class="DimmedLayer" style="background-color: transparent;"></div>
  (Session info: chrome=137.0.7151.56)
Stacktrace:
	GetHandleVerifier [0x0x78f173+62931]
	GetHandleVerifier [0x0x78f1b4+62996]
	(No symbol) [0x0x5a1053]
	(No symbol) [0x0x5efad0]
	(No symbol) [0x0x5ede8a]
	(No symbol) [0x0x5eba07]
	(No symbol) [0x0x5ead0b]
	(No symbol) [0x0x5df365]
	(No symbol) [0x0x60e4ac]
	(No symbol) [0x0x5dedf4]
	(No symbol) [0x0x60e724]
	(No symbol) [0x0x62f9e3]
	(No symbol) [0x0x60e2a6]
	(No symbol) [0x0x5dd5f0]
	(No symbol) [0x0x5de464]
	GetHandleVerifier [0x0x9e3463+2504899]
	GetHandleVerifier [0x0x9de892+2485490]
	GetHandleVerifier [0x0x7b590a+220522]
	GetHandleVerifier [0x0x7a6388+157672]
	GetHandleVerifier [0x0x7acb6d+184269]
	GetHandleVerifier [0x0x797